In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Diretórios base do projeto
project_root = Path("..").resolve()
processed_path = project_root / "data" / "processed"
output_path = project_root / "data" / "output"
output_path.mkdir(parents=True, exist_ok=True)

# Carregando dataframes processados
# Cada arquivo já foi tratado no pipeline anterior; aqui apenas consolidamos a análise.
df_orders = pd.read_csv(processed_path / "orders_processed.csv", encoding="UTF-8")
df_items = pd.read_csv(processed_path / "order_items_processed.csv", encoding="UTF-8")
df_payments = pd.read_csv(processed_path / "order_payments_processed.csv", encoding="UTF-8")
df_reviews = pd.read_csv(processed_path / "order_reviews_processed.csv", encoding="UTF-8")
df_products = pd.read_csv(processed_path / "products_processed.csv", encoding="UTF-8")
df_sellers = pd.read_csv(processed_path / "sellers_processed.csv", encoding="UTF-8")
df_customers = pd.read_csv(processed_path / "customers_processed.csv", encoding="UTF-8")

In [2]:
# Bloco 1: conversão de colunas temporais para facilitar dashboards e agregações mensais
# A padronização do tipo datetime é essencial para análise temporal e cálculo de SLA.
for df, datetime_cols in [
    (df_orders, ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]),
    (df_items, ["shipping_limit_date"]),
    (df_reviews, ["review_creation_date", "review_answer_timestamp"]),
]:
    for col in datetime_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

# Extração de componentes de data para uso em Tableau e filtros
# Isso permite responder perguntas como: receita por mês, pedidos por dia da semana e atraso por região.
df_orders = df_orders.assign(
    purchase_year=df_orders["order_purchase_timestamp"].dt.year,
    purchase_month=df_orders["order_purchase_timestamp"].dt.month,
    purchase_month_name=df_orders["order_purchase_timestamp"].dt.month_name(locale="pt_BR"),
    purchase_day=df_orders["order_purchase_timestamp"].dt.day,
    purchase_dayofweek=df_orders["order_purchase_timestamp"].dt.dayofweek,
    purchase_weekday_name=df_orders["order_purchase_timestamp"].dt.day_name(),
    purchase_hour=df_orders["order_purchase_timestamp"].dt.hour,
    approval_year=df_orders["order_approved_at"].dt.year,
    approval_month=df_orders["order_approved_at"].dt.month,
    approval_day=df_orders["order_approved_at"].dt.day,
    carrier_year=df_orders["order_delivered_carrier_date"].dt.year,
    carrier_month=df_orders["order_delivered_carrier_date"].dt.month,
    carrier_day=df_orders["order_delivered_carrier_date"].dt.day,
    delivered_year=df_orders["order_delivered_customer_date"].dt.year,
    delivered_month=df_orders["order_delivered_customer_date"].dt.month,
    delivered_day=df_orders["order_delivered_customer_date"].dt.day,
    estimated_year=df_orders["order_estimated_delivery_date"].dt.year,
    estimated_month=df_orders["order_estimated_delivery_date"].dt.month,
    estimated_day=df_orders["order_estimated_delivery_date"].dt.day,
)

# Corrigindo possível incompatibilidade de locale em alguns ambientes.
# Se necessário, a coluna de nome do mês pode ser usada sem depender do locale.
df_orders["purchase_month_name"] = df_orders["order_purchase_timestamp"].dt.strftime("%b")

# Cálculo de métricas de logística e satisfação
# Estas colunas ajudam na análise de atraso e relação com avaliações.
df_orders["delivery_days"] = (
    (df_orders["order_delivered_customer_date"] - df_orders["order_purchase_timestamp"]).dt.total_seconds() / 86400
)
df_orders["estimated_delivery_days"] = (
    (df_orders["order_estimated_delivery_date"] - df_orders["order_purchase_timestamp"]).dt.total_seconds() / 86400
)
df_orders["delay_days"] = (
    (df_orders["order_delivered_customer_date"] - df_orders["order_estimated_delivery_date"]).dt.total_seconds() / 86400
)

df_orders["is_delayed"] = (df_orders["delay_days"] > 0).astype(int)

df_orders["delay_flag"] = np.where(df_orders["delay_days"] > 0, "Atrasado", "NoPrazo")

In [3]:
# Bloco 2: agregação de pagamentos por pedido para facilitar a análise financeira
# A chave de junção é order_id; cada pedido pode ter um ou mais pagamentos.
df_payments_agg = (
    df_payments.groupby("order_id", as_index=False)
    .agg(
        payment_total=("payment_value", "sum"),
        payment_methods_count=("payment_type", "nunique"),
        payment_types=("payment_type", lambda s: ", ".join(s.dropna().astype(str).unique()))
    )
)
display(df_payments_agg)

,order_id,payment_total,payment_methods_count,payment_types
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,1,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,credit_card
...,...,...,...,...
99435,fffc94f6ce00a00581880bf54a75a037,343.40,1,boleto
99436,fffcd46ef2263f404302a634eb57f7eb,386.53,1,boleto
99437,fffce4705a9662cd70adb13d4a31832d,116.85,1,credit_card
99438,fffe18544ffabc95dfada21779c9644f,64.71,1,credit_card


In [10]:
# Bloco 3: agregação de itens por pedido para compor a visão de Receita e volume por pedido
# Isso permite calcular ticket médio, itens por pedido e valor bruto/logístico.
df_items_agg = (
    df_items.groupby("order_id", as_index=False)
    .agg(
        items_quantity=("order_item_id", "count"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum")
    )
)
display(df_items_agg)

,order_id,items_quantity,total_price,total_freight
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14
...,...,...,...,...
98661,fffc94f6ce00a00581880bf54a75a037,1,299.99,43.41
98662,fffcd46ef2263f404302a634eb57f7eb,1,350.00,36.53
98663,fffce4705a9662cd70adb13d4a31832d,1,99.90,16.95
98664,fffe18544ffabc95dfada21779c9644f,1,55.99,8.72


In [11]:
# Bloco 4: agregação de avaliações por pedido para responder relação entre qualidade e logística
# Como cada pedido pode ter eventualmente uma avaliação, usamos left join preservando pedidos sem revisão.
df_reviews_agg = (
    df_reviews.groupby("order_id", as_index=False)
    .agg(
        review_score_mean=("review_score", "mean"),
        review_score_max=("review_score", "max"),
        review_score_min=("review_score", "min"),
        review_count=("review_id", "count")
    )
)
display(df_reviews_agg)

,order_id,review_score_mean,review_score_max,review_score_min,review_count
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,5,5,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,4,4,1
2,000229ec398224ef6ca0657da4fc703e,5.0,5,5,1
3,00024acbcdf0a6daa1e931b038114c75,4.0,4,4,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,5,5,1
...,...,...,...,...,...
98668,fffc94f6ce00a00581880bf54a75a037,5.0,5,5,1
98669,fffcd46ef2263f404302a634eb57f7eb,5.0,5,5,1
98670,fffce4705a9662cd70adb13d4a31832d,5.0,5,5,1
98671,fffe18544ffabc95dfada21779c9644f,5.0,5,5,1


In [ ]:
# Bloco 5: joins principais do modelo analítico
# A abordagem de camada analítica mantém os joins organizados por responsabilidade.
# 5.1 Pedido + cliente
orders_customers = df_orders.merge(df_customers, on="customer_id", how="left", suffixes=("_order", "_customer"))
display(orders_customers)

# 5.2 Pedido + itens agregados
orders_items = orders_customers.merge(df_items_agg, on="order_id", how="left")
display(orders_items)

# 5.3 Pedido + pagamentos agregados
orders_items_payments = orders_items.merge(df_payments_agg, on="order_id", how="left")
display(orders_items_payments)

# 5.4 Pedido + avaliações agregados
orders_items_payments_reviews = orders_items_payments.merge(df_reviews_agg, on="order_id", how="left")
display(orders_items_payments_reviews)

# 5.5 Pedido + produto + vendedor
# Aqui usamos order_id como base para enriquecer com produto e vendedor de cada item.
# Como um pedido pode ter múltiplos itens, o join direto em df_items preserva a granularidade de item.
model_df = orders_items_payments_reviews.merge(df_items, on="order_id", how="left")
model_df = model_df.merge(df_products, on="product_id", how="left", suffixes=("_item", "_product"))
model_df = model_df.merge(df_sellers, on="seller_id", how="left", suffixes=("_item", "_seller"))

# Ajustes de tipos numéricos e preenchimento
model_df["payment_total"] = model_df["payment_total"].fillna(0)
model_df["total_price"] = model_df["total_price"].fillna(0)
model_df["total_freight"] = model_df["total_freight"].fillna(0)
model_df["review_score_mean"] = model_df["review_score_mean"].fillna(0)
model_df["review_count"] = model_df["review_count"].fillna(0)

# Como o schema processado não possui a coluna `quantity`, representamos a unidade do item por linha.
# Isso mantém a granularidade de item e permite calcular a receita por linha do pedido.
model_df["quantity"] = 1
model_df["item_revenue"] = model_df["price"] * model_df["quantity"]

In [7]:
# Bloco 6: exportação dos datasets prontos para o Tableau
# Os CSVs aqui são gerados para atender às perguntas de negócio do arquivo de entendimento.
model_df.to_csv(output_path / "orders_customer_product_seller_full.csv", index=False)
orders_items_payments_reviews.to_csv(output_path / "orders_customer_financial_analytics.csv", index=False)
df_orders.to_csv(output_path / "orders_temporal_features.csv", index=False)

display(model_df.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase_year,purchase_month,...,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,quantity,item_revenue
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,Delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,10,...,Utilidades_Domesticas,500.0,19.0,8.0,13.0,9350.0,Maua,SP,1,29.99
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,Delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018,7,...,Perfumaria,400.0,19.0,13.0,19.0,31570.0,Belo Horizonte,SP,1,118.70
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,Delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018,8,...,Automotivo,420.0,24.0,19.0,21.0,14840.0,Guariba,SP,1,159.90
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,Delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,2017,11,...,Pet_Shop,450.0,30.0,10.0,20.0,31842.0,Belo Horizonte,MG,1,45.00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,Delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2018,2,...,Papelaria,250.0,51.0,15.0,15.0,8752.0,Mogi Das Cruzes,SP,1,19.90
